# Siglet-Qubit Simulation: Phase 1.2

**Date:** 2025-11-28  
**Author:** Luiz Frias  
**Version:** 1.2.0

---

## Executive Summary

This notebook addresses critical brittleness identified in Phase 1 and implements four key improvements:

| Step | Description | EIG Priority |
|------|-------------|-------------|
| **A** | Restore full ε-vector structure (Pauli tensor products) | HIGH |
| **B** | Complete stability evaluation with noise sweep | HIGH |
| **C** | Hybrid enumeration + Optuna operator optimization | MEDIUM |
| **D** | 3D Hilbert/Bloch sphere projection | MEDIUM |

**Proof Gates Required:**
1. ε-vectors span orthogonal Hermitian subspace
2. Cluster stability ARI > 0.7 at σ = 0.05
3. Selected operator triple outperforms random baseline (p < 0.01)
4. Cluster centroids in Bloch space significantly separated (ANOVA p < 0.01)

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Load Phase 1 Artifacts](#2-load-phase-1-artifacts)
3. [Step A: Full ε-Vector Restoration](#3-step-a-full-ε-vector-restoration)
4. [Step B: Complete Stability Evaluation](#4-step-b-complete-stability-evaluation)
5. [Step C: Hybrid Optuna Optimization](#5-step-c-hybrid-optuna-optimization)
6. [Step D: 3D Bloch Sphere Projection](#6-step-d-3d-bloch-sphere-projection)
7. [Proof Gate Summary](#7-proof-gate-summary)
8. [Export Artifacts](#8-export-artifacts)

---

## 1. Environment Setup

In [ ]:
# Standard library
import os
import time
import json
import itertools
from pathlib import Path

# Data science
import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import pdist, squareform

# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Machine Learning
from sklearn.cluster import SpectralClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    adjusted_rand_score,
    adjusted_mutual_info_score,
)
from sklearn.preprocessing import StandardScaler

# Time series
from tslearn.metrics import cdist_dtw
from fastdtw import fastdtw

# Optimization
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Experiment tracking
import mlflow

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment loaded successfully.")

In [ ]:
# Project paths
ROOT_DIR = Path("/Users/luizfrias/CursorAI/data-science/siglet_architecture")
DATA_DIR = ROOT_DIR / "data" / "processed"
INTERIM_DIR = ROOT_DIR / "data" / "interim"
FIGURES_DIR = ROOT_DIR / "reports" / "figures"
NOTES_DIR = ROOT_DIR / "reports" / "notes"

# Create directories if needed
for d in [DATA_DIR, INTERIM_DIR, FIGURES_DIR, NOTES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# MLflow setup
mlflow.set_experiment("SigletQubit-Phase1.2")

print(f"Root: {ROOT_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Figures: {FIGURES_DIR}")

## 2. Load Phase 1 Artifacts

In [ ]:
# Load existing Phase 1 data
decay_curves = np.load(DATA_DIR / "decay_curves.npy")
curve_params = np.load(DATA_DIR / "curve_params.npy")
dtw_distances = np.load(DATA_DIR / "dtw_distances.npy")
spectral_labels = np.load(DATA_DIR / "spectral_labels.npy")
dbscan_labels = np.load(DATA_DIR / "dbscan_labels.npy")

print(f"Loaded {len(decay_curves)} decay curves")
print(f"Decay curve shape: {decay_curves.shape}")
print(f"Parameter shape: {curve_params.shape}")
print(f"DTW matrix shape: {dtw_distances.shape}")
print(f"Spectral clusters: {np.unique(spectral_labels)}")

---

## 3. Step A: Full ε-Vector Restoration

**Brittleness Identified:** Current implementation collapses ε to scalar `||ε||`, losing directional ethical information.

**Repair:** Generate Pauli tensor product basis and track evolution in full ε-space.

In [ ]:
# Define Pauli matrices
SIGMA_I = np.array([[1, 0], [0, 1]], dtype=complex)
SIGMA_X = np.array([[0, 1], [1, 0]], dtype=complex)
SIGMA_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
SIGMA_Z = np.array([[1, 0], [0, -1]], dtype=complex)

PAULI_BASIS = [SIGMA_I, SIGMA_X, SIGMA_Y, SIGMA_Z]
PAULI_NAMES = ["I", "X", "Y", "Z"]

print("Pauli matrices defined:")
for name, mat in zip(PAULI_NAMES, PAULI_BASIS):
    print(f"  σ_{name}: Hermitian={np.allclose(mat, mat.conj().T)}")

In [ ]:
def generate_pauli_tensor_basis(n=2):
    """
    Generate n-fold tensor products of Pauli matrices.

    For n=2: 4^2 = 16 operators (I⊗I, I⊗X, I⊗Y, I⊗Z, X⊗I, ...)

    Returns:
        operators: List of 4^n Hermitian operators (each is 2^n × 2^n)
        names: List of operator names (e.g., 'IX', 'XY', 'ZZ')
    """
    if n == 1:
        return PAULI_BASIS.copy(), PAULI_NAMES.copy()

    # Generate all n-fold tensor products
    operators = []
    names = []

    for indices in itertools.product(range(4), repeat=n):
        # Start with first Pauli
        op = PAULI_BASIS[indices[0]]
        name = PAULI_NAMES[indices[0]]

        # Tensor with remaining
        for idx in indices[1:]:
            op = np.kron(op, PAULI_BASIS[idx])
            name += PAULI_NAMES[idx]

        operators.append(op)
        names.append(name)

    return operators, names


# Generate 2-fold tensor basis (16 operators)
EPSILON_BASIS, EPSILON_NAMES = generate_pauli_tensor_basis(n=2)

print(f"Generated {len(EPSILON_BASIS)} ε-operators:")
print(f"  Names: {EPSILON_NAMES}")
print(f"  Operator dimension: {EPSILON_BASIS[0].shape}")

In [ ]:
# PROOF GATE A1: Verify operators are Hermitian and form orthogonal basis
with mlflow.start_run(run_name="proof_gate_A1_hermitian_orthogonality"):
    # Check Hermiticity
    hermitian_check = []
    for op, name in zip(EPSILON_BASIS, EPSILON_NAMES):
        is_hermitian = np.allclose(op, op.conj().T)
        hermitian_check.append(is_hermitian)

    all_hermitian = all(hermitian_check)
    mlflow.log_metric("all_operators_hermitian", int(all_hermitian))
    print(f"✓ All operators Hermitian: {all_hermitian}")

    # Check orthogonality via Hilbert-Schmidt inner product
    # <A, B> = Tr(A† B) / dim
    n_ops = len(EPSILON_BASIS)
    dim = EPSILON_BASIS[0].shape[0]

    inner_products = np.zeros((n_ops, n_ops))
    for i, op_i in enumerate(EPSILON_BASIS):
        for j, op_j in enumerate(EPSILON_BASIS):
            inner_products[i, j] = np.abs(np.trace(op_i.conj().T @ op_j) / dim)

    # Should be identity matrix (orthonormal)
    is_orthonormal = np.allclose(inner_products, np.eye(n_ops))
    mlflow.log_metric("operators_orthonormal", int(is_orthonormal))
    print(f"✓ Operators orthonormal: {is_orthonormal}")

    # Visualize inner product matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        inner_products,
        xticklabels=EPSILON_NAMES,
        yticklabels=EPSILON_NAMES,
        cmap="viridis",
        annot=False,
    )
    plt.title("ε-Operator Hilbert-Schmidt Inner Products")
    plt.tight_layout()

    fig_path = FIGURES_DIR / "epsilon_inner_products.png"
    plt.savefig(fig_path, dpi=300)
    mlflow.log_artifact(str(fig_path))
    plt.show()

    # Log proof gate result
    proof_gate_A1 = all_hermitian and is_orthonormal
    mlflow.log_metric("proof_gate_A1_passed", int(proof_gate_A1))
    print(f"\n{'=' * 50}")
    print(f"PROOF GATE A1: {'PASSED ✓' if proof_gate_A1 else 'FAILED ✗'}")
    print(f"{'=' * 50}")

In [ ]:
class SigletQubitFull:
    """
    Extended SigletQubit with full ε-vector support.

    Parameters:
        theta (float): Resonance angle [0, π]
        tau (float): Coherence factor [0, 1]
        epsilon_idx (int): Index into EPSILON_BASIS [0, 15]
        mu (float): Modality coherence [0, 1]
        c (float): Compression efficiency [0, 1]
    """

    def __init__(self, theta, tau, epsilon_idx, mu=0.8, c=0.7):
        self.theta = theta
        self.tau = tau
        self.epsilon_idx = epsilon_idx
        self.epsilon = EPSILON_BASIS[epsilon_idx]
        self.epsilon_name = EPSILON_NAMES[epsilon_idx]
        self.mu = mu
        self.c = c

        # Compute ε-norm (for backward compatibility)
        self.eps_norm = np.linalg.norm(self.epsilon, "fro") / np.sqrt(
            self.epsilon.shape[0]
        )

    def resonance(self):
        return np.cos(self.theta)

    def truth_score(self, t):
        """T(s,t) = cos(θ) × τ^t × ||ε|| × μ × c"""
        r = self.resonance()
        return r * (self.tau**t) * self.eps_norm * self.mu * self.c

    def truth_trajectory(self, t_max=10):
        """Generate full decay trajectory."""
        return np.array([self.truth_score(t) for t in range(t_max + 1)])

    def bloch_vector(self):
        """
        Compute Bloch-like coordinates for this siglet's ε-operator.

        For a 4×4 operator, we project onto the single-qubit Pauli basis
        by taking partial traces.
        """
        # Normalize operator
        rho = (
            self.epsilon / np.trace(self.epsilon)
            if np.trace(self.epsilon) != 0
            else self.epsilon
        )

        # Project onto Pauli basis (use first qubit)
        # Partial trace over second qubit
        dim = 2  # Single qubit dimension
        rho_reduced = np.zeros((dim, dim), dtype=complex)
        for i in range(dim):
            for j in range(dim):
                for k in range(dim):
                    rho_reduced[i, j] += rho[i * dim + k, j * dim + k]

        # Bloch coordinates
        bx = np.trace(rho_reduced @ SIGMA_X).real
        by = np.trace(rho_reduced @ SIGMA_Y).real
        bz = np.trace(rho_reduced @ SIGMA_Z).real

        return np.array([bx, by, bz])

    def __repr__(self):
        return f"SigletQubitFull(θ={self.theta:.3f}, τ={self.tau:.3f}, ε={self.epsilon_name})"

In [ ]:
# Generate siglets with full ε-vector structure
with mlflow.start_run(run_name="generate_full_epsilon_siglets"):
    start_time = time.time()

    # Parameters
    N_SIGLETS = 1000
    T_MAX = 10

    # Sample parameters
    thetas = np.linspace(0, np.pi, 50)
    taus = np.linspace(0.8, 1.0, 20)

    mlflow.log_params(
        {
            "n_siglets": N_SIGLETS,
            "t_max": T_MAX,
            "n_epsilon_operators": len(EPSILON_BASIS),
            "theta_range": [0, np.pi],
            "tau_range": [0.8, 1.0],
        }
    )

    # Generate siglets
    siglets = []
    decay_curves_full = []
    epsilon_indices = []
    bloch_vectors = []
    params_full = []

    for i, theta in enumerate(thetas):
        for j, tau in enumerate(taus):
            # Assign ε-operator based on position in parameter space
            # This creates structured assignment for analysis
            eps_idx = (i + j) % len(EPSILON_BASIS)

            sq = SigletQubitFull(theta=theta, tau=tau, epsilon_idx=eps_idx)
            siglets.append(sq)

            decay_curves_full.append(sq.truth_trajectory(T_MAX))
            epsilon_indices.append(eps_idx)
            bloch_vectors.append(sq.bloch_vector())
            params_full.append([theta, tau, eps_idx, sq.mu, sq.c])

    # Convert to arrays
    decay_curves_full = np.array(decay_curves_full)
    epsilon_indices = np.array(epsilon_indices)
    bloch_vectors = np.array(bloch_vectors)
    params_full = np.array(params_full)

    # Log metrics
    computation_time = time.time() - start_time
    mlflow.log_metric("computation_time", computation_time)
    mlflow.log_metric("n_curves_generated", len(decay_curves_full))

    print(f"Generated {len(siglets)} siglets in {computation_time:.2f}s")
    print(f"Decay curves shape: {decay_curves_full.shape}")
    print(f"Bloch vectors shape: {bloch_vectors.shape}")
    print(f"ε-operator distribution: {np.bincount(epsilon_indices)}")

---

## 4. Step B: Complete Stability Evaluation

**Brittleness Identified:** Noise resilience testing incomplete; no formal cluster persistence metric.

**Repair:** Systematic noise sweep with ARI tracking and bootstrap confidence intervals.

In [ ]:
def compute_dtw_matrix_fast(curves, progress=True):
    """
    Compute DTW distance matrix using fastdtw for efficiency.
    """
    n = len(curves)
    dist_matrix = np.zeros((n, n))

    total = n * (n - 1) // 2
    count = 0

    for i in range(n):
        for j in range(i + 1, n):
            distance, _ = fastdtw(curves[i], curves[j])
            dist_matrix[i, j] = distance
            dist_matrix[j, i] = distance
            count += 1

            if progress and count % 10000 == 0:
                print(f"  Progress: {count}/{total} ({100 * count / total:.1f}%)")

    return dist_matrix

In [ ]:
def evaluate_cluster_stability(
    curves, base_labels, noise_levels, n_clusters=5, n_bootstrap=50
):
    """
    Evaluate cluster stability across noise perturbations.

    Returns:
        results: Dict with ARI scores and statistics for each noise level
    """
    results = {
        "noise_levels": noise_levels,
        "mean_ari": [],
        "std_ari": [],
        "min_ari": [],
        "max_ari": [],
        "all_ari": [],
    }

    for noise_level in noise_levels:
        print(f"  Testing noise level σ={noise_level}...")
        ari_scores = []

        for bootstrap_iter in range(n_bootstrap):
            # Add Gaussian noise to curves
            noisy_curves = curves + np.random.normal(0, noise_level, curves.shape)

            # Compute DTW on noisy curves (use sampling for speed)
            sample_size = min(200, len(curves))
            sample_idx = np.random.choice(len(curves), sample_size, replace=False)

            noisy_sample = noisy_curves[sample_idx]
            base_sample_labels = base_labels[sample_idx]

            # Compute DTW matrix for sample
            dtw_sample = np.zeros((sample_size, sample_size))
            for i in range(sample_size):
                for j in range(i + 1, sample_size):
                    d, _ = fastdtw(noisy_sample[i], noisy_sample[j])
                    dtw_sample[i, j] = d
                    dtw_sample[j, i] = d

            # Normalize and cluster
            dtw_norm = dtw_sample / (np.max(dtw_sample) + 1e-10)
            affinity = np.exp(-dtw_norm)

            try:
                clustering = SpectralClustering(
                    n_clusters=n_clusters,
                    affinity="precomputed",
                    random_state=RANDOM_SEED + bootstrap_iter,
                )
                noisy_labels = clustering.fit_predict(affinity)

                # Compute ARI
                ari = adjusted_rand_score(base_sample_labels, noisy_labels)
                ari_scores.append(ari)
            except Exception as e:
                print(f"    Bootstrap {bootstrap_iter} failed: {e}")
                continue

        if ari_scores:
            results["mean_ari"].append(np.mean(ari_scores))
            results["std_ari"].append(np.std(ari_scores))
            results["min_ari"].append(np.min(ari_scores))
            results["max_ari"].append(np.max(ari_scores))
            results["all_ari"].append(ari_scores)
        else:
            results["mean_ari"].append(0)
            results["std_ari"].append(0)
            results["min_ari"].append(0)
            results["max_ari"].append(0)
            results["all_ari"].append([])

    return results

In [ ]:
# Run stability evaluation on NEW Phase 1.2 decay curves
with mlflow.start_run(run_name="stability_evaluation"):
    print("Running stability evaluation on Phase 1.2 curves...")
    start_time = time.time()

    # Noise levels to test
    NOISE_LEVELS = [0.01, 0.02, 0.05, 0.1, 0.2]
    N_BOOTSTRAP = 30  # Reduced for speed
    N_CLUSTERS = 5

    mlflow.log_params(
        {
            "noise_levels": NOISE_LEVELS,
            "n_bootstrap": N_BOOTSTRAP,
            "n_clusters": N_CLUSTERS,
            "data_source": "decay_curves_full (Phase 1.2)",
        }
    )

    # STEP 1: Compute DTW matrix for the NEW decay_curves_full
    # This is necessary to establish baseline clustering labels
    print("  Computing DTW matrix for Phase 1.2 curves...")
    sample_size_dtw = min(300, len(decay_curves_full))  # Sample for efficiency
    sample_idx_dtw = np.random.choice(
        len(decay_curves_full), sample_size_dtw, replace=False
    )
    decay_curves_sample = decay_curves_full[sample_idx_dtw]

    dtw_matrix_full = np.zeros((sample_size_dtw, sample_size_dtw))
    for i in range(sample_size_dtw):
        for j in range(i + 1, sample_size_dtw):
            d, _ = fastdtw(decay_curves_sample[i], decay_curves_sample[j])
            dtw_matrix_full[i, j] = d
            dtw_matrix_full[j, i] = d
    print(f"    DTW matrix computed: {dtw_matrix_full.shape}")

    # STEP 2: Compute spectral clustering labels for the new curves
    print("  Computing spectral clustering labels...")
    dtw_norm = dtw_matrix_full / (np.max(dtw_matrix_full) + 1e-10)
    affinity_full = np.exp(-dtw_norm)

    clustering_full = SpectralClustering(
        n_clusters=N_CLUSTERS, affinity="precomputed", random_state=RANDOM_SEED
    )
    spectral_labels_full = clustering_full.fit_predict(affinity_full)
    print(f"    Cluster distribution: {np.bincount(spectral_labels_full)}")

    # STEP 3: Run stability evaluation on the NEW Phase 1.2 curves
    print("  Running stability evaluation...")
    stability_results = evaluate_cluster_stability(
        curves=decay_curves_sample,  # Use the sampled Phase 1.2 curves
        base_labels=spectral_labels_full,  # Use the NEW labels
        noise_levels=NOISE_LEVELS,
        n_clusters=N_CLUSTERS,
        n_bootstrap=N_BOOTSTRAP,
    )

    # Log results
    for i, noise in enumerate(NOISE_LEVELS):
        mlflow.log_metric(f"ari_mean_noise_{noise}", stability_results["mean_ari"][i])
        mlflow.log_metric(f"ari_std_noise_{noise}", stability_results["std_ari"][i])

    computation_time = time.time() - start_time
    mlflow.log_metric("stability_computation_time", computation_time)

    print(f"\nStability evaluation completed in {computation_time:.2f}s")
    print("\nResults:")
    print("-" * 50)
    for i, noise in enumerate(NOISE_LEVELS):
        print(
            f"  σ={noise:.2f}: ARI = {stability_results['mean_ari'][i]:.4f} ± {stability_results['std_ari'][i]:.4f}"
        )

In [ ]:
# PROOF GATE B: Cluster stability at σ=0.05
with mlflow.start_run(run_name="proof_gate_B_stability"):
    # Find ARI at σ=0.05
    target_noise = 0.05
    target_idx = NOISE_LEVELS.index(target_noise)
    ari_at_target = stability_results["mean_ari"][target_idx]

    # Threshold
    ARI_THRESHOLD = 0.7
    proof_gate_B = ari_at_target > ARI_THRESHOLD

    mlflow.log_metric("ari_at_005", ari_at_target)
    mlflow.log_metric("ari_threshold", ARI_THRESHOLD)
    mlflow.log_metric("proof_gate_B_passed", int(proof_gate_B))

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot 1: ARI vs Noise Level
    ax1 = axes[0]
    ax1.errorbar(
        NOISE_LEVELS,
        stability_results["mean_ari"],
        yerr=stability_results["std_ari"],
        marker="o",
        capsize=5,
        linewidth=2,
        markersize=8,
    )
    ax1.axhline(
        y=ARI_THRESHOLD, color="r", linestyle="--", label=f"Threshold ({ARI_THRESHOLD})"
    )
    ax1.axvline(
        x=target_noise,
        color="g",
        linestyle=":",
        alpha=0.7,
        label=f"Target σ={target_noise}",
    )
    ax1.set_xlabel("Noise Level (σ)")
    ax1.set_ylabel("Adjusted Rand Index (ARI)")
    ax1.set_title("Cluster Stability Under Noise Perturbation")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1])

    # Plot 2: ARI Distribution at σ=0.05
    ax2 = axes[1]
    if stability_results["all_ari"][target_idx]:
        ax2.hist(
            stability_results["all_ari"][target_idx],
            bins=15,
            edgecolor="black",
            alpha=0.7,
        )
        ax2.axvline(
            x=ARI_THRESHOLD,
            color="r",
            linestyle="--",
            label=f"Threshold ({ARI_THRESHOLD})",
        )
        ax2.axvline(
            x=ari_at_target,
            color="g",
            linestyle="-",
            linewidth=2,
            label=f"Mean ({ari_at_target:.3f})",
        )
    ax2.set_xlabel("ARI")
    ax2.set_ylabel("Frequency")
    ax2.set_title(f"ARI Distribution at σ={target_noise}")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()

    fig_path = FIGURES_DIR / "stability_sweep.png"
    plt.savefig(fig_path, dpi=300)
    mlflow.log_artifact(str(fig_path))
    plt.show()

    print(f"\n{'=' * 50}")
    print(f"PROOF GATE B: {'PASSED ✓' if proof_gate_B else 'FAILED ✗'}")
    print(f"  ARI at σ=0.05: {ari_at_target:.4f} (threshold: {ARI_THRESHOLD})")
    print(f"{'=' * 50}")

---

## 5. Step C: Hybrid Optuna Optimization

**Brittleness Identified:** Section Two stubs exist but no executable implementation.

**Repair:** Enumerate operator triples, score by composite metric, optimize with Optuna.

In [ ]:
def compute_commutation(op1, op2):
    """
    Compute the Frobenius norm of the commutator [A, B] = AB - BA.
    Lower values indicate better commutativity.
    """
    commutator = op1 @ op2 - op2 @ op1
    return np.linalg.norm(commutator, "fro")


def evaluate_operator_triple(triple_indices, siglets_sample, n_clusters=5):
    """
    Evaluate a triple of operators based on:
    1. Commutation sum (lower is better)
    2. Cluster separation (higher is better)
    3. Noise resilience (higher is better)

    Returns:
        scores: Dict with individual and composite scores
    """
    op1, op2, op3 = [EPSILON_BASIS[i] for i in triple_indices]

    # 1. Commutation sum (normalized to [0, 1], inverted so higher is better)
    comm_12 = compute_commutation(op1, op2)
    comm_13 = compute_commutation(op1, op3)
    comm_23 = compute_commutation(op2, op3)
    comm_sum = comm_12 + comm_13 + comm_23

    # Normalize (typical max is ~8 for 4x4 matrices)
    comm_score = 1.0 - min(1.0, comm_sum / 24.0)

    # 2. Cluster separation (using silhouette on a sample)
    # Filter siglets that use these operators
    mask = np.isin(siglets_sample["epsilon_idx"], triple_indices)
    if np.sum(mask) < n_clusters * 2:
        # Not enough samples
        return {
            "comm_score": comm_score,
            "cluster_score": 0,
            "composite": comm_score * 0.5,
        }

    filtered_curves = siglets_sample["curves"][mask]

    if len(filtered_curves) < 10:
        return {
            "comm_score": comm_score,
            "cluster_score": 0,
            "composite": comm_score * 0.5,
        }

    # Quick DTW and clustering
    try:
        dtw_mat = np.zeros((len(filtered_curves), len(filtered_curves)))
        for i in range(len(filtered_curves)):
            for j in range(i + 1, len(filtered_curves)):
                d, _ = fastdtw(filtered_curves[i], filtered_curves[j])
                dtw_mat[i, j] = d
                dtw_mat[j, i] = d

        dtw_norm = dtw_mat / (np.max(dtw_mat) + 1e-10)
        affinity = np.exp(-dtw_norm)

        actual_clusters = min(n_clusters, len(filtered_curves) // 2)
        if actual_clusters < 2:
            return {
                "comm_score": comm_score,
                "cluster_score": 0,
                "composite": comm_score * 0.5,
            }

        clustering = SpectralClustering(
            n_clusters=actual_clusters, affinity="precomputed", random_state=RANDOM_SEED
        )
        labels = clustering.fit_predict(affinity)

        if len(np.unique(labels)) > 1:
            sil_score = silhouette_score(dtw_mat, labels, metric="precomputed")
            cluster_score = (sil_score + 1) / 2  # Normalize to [0, 1]
        else:
            cluster_score = 0
    except Exception:
        cluster_score = 0

    # Composite score
    composite = 0.5 * comm_score + 0.5 * cluster_score

    return {
        "comm_score": comm_score,
        "cluster_score": cluster_score,
        "composite": composite,
    }

In [ ]:
# Enumerate all operator triples and compute quick proxy scores
with mlflow.start_run(run_name="operator_triple_enumeration"):
    print("Enumerating all operator triples...")
    start_time = time.time()

    # Generate all C(16, 3) = 560 triples
    all_triples = list(itertools.combinations(range(len(EPSILON_BASIS)), 3))
    print(f"Total triples: {len(all_triples)}")

    mlflow.log_param("total_triples", len(all_triples))

    # Quick proxy: commutation score only
    proxy_scores = []
    for triple in all_triples:
        op1, op2, op3 = [EPSILON_BASIS[i] for i in triple]
        comm_sum = (
            compute_commutation(op1, op2)
            + compute_commutation(op1, op3)
            + compute_commutation(op2, op3)
        )
        proxy_scores.append(1.0 - min(1.0, comm_sum / 24.0))

    # Rank triples
    triple_rankings = sorted(zip(all_triples, proxy_scores), key=lambda x: -x[1])

    # Top K for full evaluation
    TOP_K = 50
    top_triples = [t for t, s in triple_rankings[:TOP_K]]

    print(f"\nTop {TOP_K} triples by commutation score:")
    for i, (triple, score) in enumerate(triple_rankings[:10]):
        names = [EPSILON_NAMES[idx] for idx in triple]
        print(f"  {i + 1}. {names}: {score:.4f}")

    mlflow.log_metric("top_k_selected", TOP_K)
    mlflow.log_metric("best_proxy_score", triple_rankings[0][1])

    computation_time = time.time() - start_time
    mlflow.log_metric("enumeration_time", computation_time)
    print(f"\nEnumeration completed in {computation_time:.2f}s")

In [ ]:
# Optuna optimization on top-K triples
with mlflow.start_run(run_name="optuna_triple_optimization"):
    print("Running Optuna optimization on top triples...")
    start_time = time.time()

    # Prepare sample data for evaluation
    siglets_sample = {
        "curves": decay_curves_full,
        "epsilon_idx": epsilon_indices,
        "params": params_full,
    }

    def objective(trial):
        # Select triple from top-K
        triple_idx = trial.suggest_int("triple_idx", 0, len(top_triples) - 1)
        triple = top_triples[triple_idx]

        # Evaluate
        scores = evaluate_operator_triple(triple, siglets_sample)

        return scores["composite"]

    # Run optimization
    N_TRIALS = 100  # Reduced for speed
    mlflow.log_param("n_trials", N_TRIALS)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    # Best result
    best_triple_idx = study.best_params["triple_idx"]
    best_triple = top_triples[best_triple_idx]
    best_names = [EPSILON_NAMES[idx] for idx in best_triple]
    best_score = study.best_value

    mlflow.log_params(
        {
            "best_triple": str(best_triple),
            "best_triple_names": str(best_names),
        }
    )
    mlflow.log_metric("best_composite_score", best_score)

    computation_time = time.time() - start_time
    mlflow.log_metric("optuna_time", computation_time)

    print(f"\nOptuna optimization completed in {computation_time:.2f}s")
    print(f"\nBest Triple: {best_names}")
    print(f"Best Composite Score: {best_score:.4f}")

In [ ]:
# PROOF GATE C: Compare best triple to random baseline
with mlflow.start_run(run_name="proof_gate_C_optuna_significance"):
    print("Testing significance vs random baseline...")

    # Generate random baseline scores
    N_RANDOM = 100
    random_scores = []

    for _ in range(N_RANDOM):
        random_triple = tuple(np.random.choice(len(EPSILON_BASIS), 3, replace=False))
        scores = evaluate_operator_triple(random_triple, siglets_sample)
        random_scores.append(scores["composite"])

    random_scores = np.array(random_scores)

    # Statistical test: is best_score significantly better than random?
    # Use one-sample t-test against best score
    t_stat, p_value = stats.ttest_1samp(random_scores, best_score)

    # Also compute percentile rank
    percentile = stats.percentileofscore(random_scores, best_score)

    mlflow.log_metrics(
        {
            "random_baseline_mean": np.mean(random_scores),
            "random_baseline_std": np.std(random_scores),
            "best_vs_random_t_stat": t_stat,
            "best_vs_random_p_value": p_value,
            "best_percentile_rank": percentile,
        }
    )

    # Proof gate: p < 0.01 and best > mean(random)
    proof_gate_C = (p_value < 0.01) and (best_score > np.mean(random_scores))
    mlflow.log_metric("proof_gate_C_passed", int(proof_gate_C))

    # Visualization
    plt.figure(figsize=(10, 6))
    plt.hist(
        random_scores, bins=20, edgecolor="black", alpha=0.7, label="Random Baseline"
    )
    plt.axvline(
        x=best_score,
        color="r",
        linestyle="-",
        linewidth=2,
        label=f"Best Triple ({best_score:.3f})",
    )
    plt.axvline(
        x=np.mean(random_scores),
        color="g",
        linestyle="--",
        label=f"Random Mean ({np.mean(random_scores):.3f})",
    )
    plt.xlabel("Composite Score")
    plt.ylabel("Frequency")
    plt.title(f"Best Triple vs Random Baseline (p={p_value:.4f})")
    plt.legend()
    plt.grid(True, alpha=0.3)

    fig_path = FIGURES_DIR / "optuna_vs_random.png"
    plt.savefig(fig_path, dpi=300)
    mlflow.log_artifact(str(fig_path))
    plt.show()

    print(f"\n{'=' * 50}")
    print(f"PROOF GATE C: {'PASSED ✓' if proof_gate_C else 'FAILED ✗'}")
    print(f"  Best score: {best_score:.4f}")
    print(f"  Random mean: {np.mean(random_scores):.4f} ± {np.std(random_scores):.4f}")
    print(f"  p-value: {p_value:.6f}")
    print(f"  Percentile rank: {percentile:.1f}%")
    print(f"{'=' * 50}")

---

## 6. Step D: 3D Bloch Sphere Projection

**Brittleness Identified:** No visualization of ethical primitives in geometric space.

**Repair:** Project ε-operators onto Bloch sphere, visualize cluster relationships.

In [ ]:
# Compute Bloch coordinates for all ε-operators
with mlflow.start_run(run_name="bloch_projection"):
    print("Computing Bloch coordinates for ε-operators...")

    operator_bloch = []
    for i, (op, name) in enumerate(zip(EPSILON_BASIS, EPSILON_NAMES)):
        # Normalize operator to density-matrix-like form
        trace_val = np.trace(op)
        if np.abs(trace_val) > 1e-10:
            rho = op / trace_val
        else:
            rho = op / (np.linalg.norm(op, "fro") + 1e-10)

        # Partial trace to get single-qubit reduced state
        dim = 2
        rho_reduced = np.zeros((dim, dim), dtype=complex)
        for ii in range(dim):
            for jj in range(dim):
                for kk in range(dim):
                    rho_reduced[ii, jj] += rho[ii * dim + kk, jj * dim + kk]

        # Bloch coordinates
        bx = np.trace(rho_reduced @ SIGMA_X).real
        by = np.trace(rho_reduced @ SIGMA_Y).real
        bz = np.trace(rho_reduced @ SIGMA_Z).real

        operator_bloch.append([bx, by, bz])

    operator_bloch = np.array(operator_bloch)

    print(f"Bloch coordinates computed: {operator_bloch.shape}")
    print("\nOperator Bloch coordinates:")
    for name, coords in zip(EPSILON_NAMES, operator_bloch):
        print(f"  {name}: ({coords[0]:.3f}, {coords[1]:.3f}, {coords[2]:.3f})")

In [ ]:
# Cluster ε-operators in Bloch space
with mlflow.start_run(run_name="bloch_clustering"):
    # Cluster the Bloch coordinates
    n_op_clusters = 4  # Expect 4 groups based on first Pauli index

    # Use K-means on Bloch coordinates
    from sklearn.cluster import KMeans

    kmeans_bloch = KMeans(n_clusters=n_op_clusters, random_state=RANDOM_SEED, n_init=10)
    bloch_labels = kmeans_bloch.fit_predict(operator_bloch)

    # Cluster centroids
    centroids = kmeans_bloch.cluster_centers_

    mlflow.log_metric("n_bloch_clusters", n_op_clusters)

    print(f"Cluster labels: {bloch_labels}")
    print(f"\nCluster centroids:")
    for i, c in enumerate(centroids):
        print(f"  Cluster {i}: ({c[0]:.3f}, {c[1]:.3f}, {c[2]:.3f})")

In [ ]:
# PROOF GATE D: Test cluster separation in Bloch space (ANOVA)
with mlflow.start_run(run_name="proof_gate_D_bloch_separation"):
    # ANOVA test for each Bloch coordinate
    groups = [operator_bloch[bloch_labels == i] for i in range(n_op_clusters)]

    # Test X coordinate
    f_x, p_x = stats.f_oneway(*[g[:, 0] for g in groups if len(g) > 0])
    # Test Y coordinate
    f_y, p_y = stats.f_oneway(*[g[:, 1] for g in groups if len(g) > 0])
    # Test Z coordinate
    f_z, p_z = stats.f_oneway(*[g[:, 2] for g in groups if len(g) > 0])

    # Combined p-value (Bonferroni correction)
    min_p = min(p_x, p_y, p_z)
    combined_significant = min_p < (0.01 / 3)  # Bonferroni

    mlflow.log_metrics(
        {
            "anova_f_x": f_x,
            "anova_p_x": p_x,
            "anova_f_y": f_y,
            "anova_p_y": p_y,
            "anova_f_z": f_z,
            "anova_p_z": p_z,
            "min_p_value": min_p,
        }
    )

    proof_gate_D = combined_significant
    mlflow.log_metric("proof_gate_D_passed", int(proof_gate_D))

    print(f"ANOVA Results:")
    print(f"  X-axis: F={f_x:.2f}, p={p_x:.6f}")
    print(f"  Y-axis: F={f_y:.2f}, p={p_y:.6f}")
    print(f"  Z-axis: F={f_z:.2f}, p={p_z:.6f}")
    print(f"  Min p-value: {min_p:.6f}")
    print(f"\n{'=' * 50}")
    print(f"PROOF GATE D: {'PASSED ✓' if proof_gate_D else 'FAILED ✗'}")
    print(f"{'=' * 50}")

In [ ]:
# 3D Visualization of Bloch sphere with clusters
with mlflow.start_run(run_name="bloch_visualization"):
    fig = plt.figure(figsize=(14, 12))
    ax = fig.add_subplot(111, projection="3d")

    # Plot unit sphere wireframe
    u = np.linspace(0, 2 * np.pi, 30)
    v = np.linspace(0, np.pi, 20)
    x_sphere = np.outer(np.cos(u), np.sin(v))
    y_sphere = np.outer(np.sin(u), np.sin(v))
    z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_wireframe(x_sphere, y_sphere, z_sphere, alpha=0.1, color="gray")

    # Color map for clusters
    colors = cm.viridis(np.linspace(0, 1, n_op_clusters))

    # Plot operators by cluster
    for cluster_id in range(n_op_clusters):
        mask = bloch_labels == cluster_id
        cluster_points = operator_bloch[mask]
        cluster_names = [EPSILON_NAMES[i] for i in range(len(EPSILON_NAMES)) if mask[i]]

        ax.scatter(
            cluster_points[:, 0],
            cluster_points[:, 1],
            cluster_points[:, 2],
            c=[colors[cluster_id]],
            s=100,
            label=f"Cluster {cluster_id}",
            alpha=0.8,
        )

        # Add labels
        for point, name in zip(cluster_points, cluster_names):
            ax.text(point[0], point[1], point[2], f"  {name}", fontsize=8)

    # Plot centroids
    ax.scatter(
        centroids[:, 0],
        centroids[:, 1],
        centroids[:, 2],
        c="red",
        s=200,
        marker="*",
        label="Centroids",
    )

    ax.set_xlabel("Bloch X (σₓ)")
    ax.set_ylabel("Bloch Y (σᵧ)")
    ax.set_zlabel("Bloch Z (σᵤ)")
    ax.set_title("ε-Operators in Bloch Space")
    ax.legend(loc="upper left")

    # Set equal aspect ratio
    max_range = 1.5
    ax.set_xlim([-max_range, max_range])
    ax.set_ylim([-max_range, max_range])
    ax.set_zlim([-max_range, max_range])

    plt.tight_layout()

    fig_path = FIGURES_DIR / "bloch_sphere_clusters.png"
    plt.savefig(fig_path, dpi=300)
    mlflow.log_artifact(str(fig_path))
    plt.show()

---

## 7. Proof Gate Summary

In [ ]:
# Final summary of all proof gates
with mlflow.start_run(run_name="proof_gate_summary"):
    proof_gates = {
        "A1": ("ε-operators Hermitian & Orthonormal", proof_gate_A1),
        "B": (f"Cluster stability ARI > 0.7 at σ=0.05", proof_gate_B),
        "C": ("Best triple outperforms random (p < 0.01)", proof_gate_C),
        "D": ("Bloch cluster separation (ANOVA p < 0.01)", proof_gate_D),
    }

    print("\n" + "=" * 70)
    print("PHASE 1.2 PROOF GATE SUMMARY")
    print("=" * 70)

    all_passed = True
    for gate_id, (description, passed) in proof_gates.items():
        status = "PASSED ✓" if passed else "FAILED ✗"
        print(f"  Gate {gate_id}: {status}")
        print(f"           {description}")
        mlflow.log_metric(f"proof_gate_{gate_id}", int(passed))
        if not passed:
            all_passed = False

    print("=" * 70)
    overall_status = "ALL GATES PASSED ✓" if all_passed else "SOME GATES FAILED ✗"
    print(f"OVERALL: {overall_status}")
    print("=" * 70)

    mlflow.log_metric("all_proof_gates_passed", int(all_passed))

---

## 8. Export Artifacts

In [ ]:
# Save all artifacts
with mlflow.start_run(run_name="export_artifacts"):
    print("Exporting artifacts...")

    # 1. CRITICAL: Save the new decay curves with full ε-vector structure
    # Phase 2 should load these instead of the old Phase 1 curves
    np.save(DATA_DIR / "decay_curves_phase1.2.npy", decay_curves_full)
    print(f"  Saved: decay_curves_phase1.2.npy (shape: {decay_curves_full.shape})")

    # 2. Save corresponding parameters
    np.save(DATA_DIR / "curve_params_phase1.2.npy", params_full)
    print(f"  Saved: curve_params_phase1.2.npy (shape: {params_full.shape})")

    # 3. Save ε-operator indices for each siglet
    np.save(DATA_DIR / "epsilon_indices.npy", epsilon_indices)
    print(f"  Saved: epsilon_indices.npy (shape: {epsilon_indices.shape})")

    # 4. Full ε-operator basis (flattened)
    np.save(
        DATA_DIR / "epsilon_vectors.npy",
        np.array([op.flatten() for op in EPSILON_BASIS]),
    )
    print(f"  Saved: epsilon_vectors.npy")

    # 5. Bloch coordinates
    np.save(DATA_DIR / "bloch_coordinates.npy", operator_bloch)
    print(f"  Saved: bloch_coordinates.npy")

    # 6. Optuna results
    optuna_results = pd.DataFrame(
        {
            "triple": [str(t) for t, _ in triple_rankings[:TOP_K]],
            "proxy_score": [s for _, s in triple_rankings[:TOP_K]],
            "is_best": [str(t) == str(best_triple) for t, _ in triple_rankings[:TOP_K]],
        }
    )
    optuna_results.to_csv(DATA_DIR / "optuna_triple_results.csv", index=False)
    print(f"  Saved: optuna_triple_results.csv")

    # 7. Stability results
    stability_df = pd.DataFrame(
        {
            "noise_level": NOISE_LEVELS,
            "mean_ari": stability_results["mean_ari"],
            "std_ari": stability_results["std_ari"],
        }
    )
    stability_df.to_csv(DATA_DIR / "stability_results.csv", index=False)
    print(f"  Saved: stability_results.csv")

    # 8. Summary markdown
    summary_md = f"""
# Phase 1.2 Summary

**Date:** {time.strftime("%Y-%m-%d %H:%M:%S")}

## Proof Gate Results

| Gate | Description | Status |
|------|-------------|--------|
| A1 | ε-operators Hermitian & Orthonormal | {"PASSED" if proof_gate_A1 else "FAILED"} |
| B | Cluster stability ARI > 0.7 at σ=0.05 | {"PASSED" if proof_gate_B else "FAILED"} |
| C | Best triple outperforms random (p < 0.01) | {"PASSED" if proof_gate_C else "FAILED"} |
| D | Bloch cluster separation (ANOVA p < 0.01) | {"PASSED" if proof_gate_D else "FAILED"} |

## Key Findings

### Best Operator Triple
- **Operators:** {best_names}
- **Composite Score:** {best_score:.4f}

### Stability Analysis
- **ARI at σ=0.05:** {ari_at_target:.4f}
- **Threshold:** {ARI_THRESHOLD}

### Bloch Space Analysis
- **Clusters identified:** {n_op_clusters}
- **ANOVA significance:** p = {min_p:.6f}

## Artifacts Generated

- `decay_curves_phase1.2.npy` - **New decay curves with full ε-vector structure (for Phase 2)**
- `curve_params_phase1.2.npy` - Parameters for Phase 1.2 siglets
- `epsilon_indices.npy` - ε-operator index for each siglet
- `epsilon_vectors.npy` - Full 16-dim ε-operator basis
- `bloch_coordinates.npy` - Bloch sphere projections
- `optuna_triple_results.csv` - Operator triple rankings
- `stability_results.csv` - Noise resilience data
- `bloch_sphere_clusters.png` - 3D visualization
- `stability_sweep.png` - ARI vs noise plot

## Next Steps

Proceed to **Phase 2: Falsifiability Protocols** to validate emergent structure against null baselines.
"""

    with open(NOTES_DIR / "phase1.2_summary.md", "w") as f:
        f.write(summary_md)
    print(f"  Saved: phase1.2_summary.md")

    # Log all artifacts to MLflow
    for artifact in [
        "decay_curves_phase1.2.npy",
        "curve_params_phase1.2.npy",
        "epsilon_indices.npy",
        "epsilon_vectors.npy",
        "bloch_coordinates.npy",
        "optuna_triple_results.csv",
        "stability_results.csv",
    ]:
        mlflow.log_artifact(str(DATA_DIR / artifact))
    mlflow.log_artifact(str(NOTES_DIR / "phase1.2_summary.md"))

    print("\nAll artifacts exported successfully.")

---

## End of Phase 1.2

This notebook has addressed the brittleness identified in Phase 1:

1. **Step A:** Restored full ε-vector structure using Pauli tensor products
2. **Step B:** Completed stability evaluation with systematic noise sweep
3. **Step C:** Implemented hybrid Optuna optimization for operator triples
4. **Step D:** Projected ε-operators onto Bloch sphere for geometric analysis

**Next:** Proceed to Phase 2 for falsifiability protocols and null hypothesis testing.